In [21]:
import sys, os
import numpy as np
import pandas as pd
from pathlib import Path

# Locate superligaen/ regardless of where Jupyter was launched
_here = Path.cwd()
if not (_here / 'run_model.py').exists():
    for _c in [_here / 'superligaen',
               _here / 'non_penalty_bayes' / 'superligaen',
               _here / 'team_strength' / 'non_penalty_bayes' / 'superligaen']:
        if _c.exists():
            os.chdir(_c)
            break

NOTEBOOK_DIR = Path.cwd().resolve()
NP_BAYES_DIR = str(NOTEBOOK_DIR.parent)
sys.path.insert(0, NP_BAYES_DIR)

from src.data_utils import load_and_process_data
from src.superliga_model import build_and_sample_model

LEAGUE = 'Superligaen'
SEASON = '2025-2026'

REPO_ROOT = str(NOTEBOOK_DIR.parents[4])
DB_PATH   = os.path.join(REPO_ROOT, 'infra', 'data', 'db', 'fotmob.db')

DECAY_RATE   = 0.0018
GOALS_WEIGHT = 0.35
XG_WEIGHT    = 0.50
PSXG_WEIGHT  = 0.15
EPV_WEIGHT   = 0.0
N_SAMPLES    = 10_000
N_TUNE       = 5_000

# Non-penalty model: add back penalty baseline as a fixed offset.
# Rates are penalties per game × conversion rate.
BASELINE_HOME_PENS = 0.18 * 0.78
BASELINE_AWAY_PENS = 0.13 * 0.78

print(f'CWD: {NOTEBOOK_DIR}')
print(f'DB:  {DB_PATH}')

CWD: /Users/admin/dev/algobetting/algo/models/team_strength/non_penalty_bayes/superligaen
DB:  /Users/admin/dev/algobetting/infra/data/db/fotmob.db


In [22]:
df, team_mapping, n_teams = load_and_process_data(
    db_path=DB_PATH, league=LEAGUE, season=SEASON,
    decay_rate=DECAY_RATE,
    goals_weight=GOALS_WEIGHT, xg_weight=XG_WEIGHT,
    psxg_weight=PSXG_WEIGHT, epv_weight=EPV_WEIGHT,
)
print(f'{n_teams} teams, {df["match_id"].nunique()} matches')

_, trace = build_and_sample_model(
    df, n_teams, trace=N_SAMPLES, tune=N_TUNE, team_mapping=team_mapping,
)
print('Done.')

12 teams, 162 matches
Done.


In [23]:
# ── Strength ratings table ────────────────────────────────────────────────
posterior  = trace.posterior
team_names = list(team_mapping.keys())
n_teams    = len(team_names)

att  = posterior['att_str'].values.reshape(-1, n_teams)
defn = posterior['def_str'].values.reshape(-1, n_teams)
hadv = posterior['home_adv'].values.reshape(-1)
base = posterior['baseline'].values.reshape(-1)

ratings = pd.DataFrame({
    'team':     team_names,
    'att_mean': att.mean(axis=0).round(3),
    'att_sd':   att.std(axis=0).round(3),
    'att_lo':   np.percentile(att, 5,  axis=0).round(3),
    'att_hi':   np.percentile(att, 95, axis=0).round(3),
    'def_mean': defn.mean(axis=0).round(3),
    'def_sd':   defn.std(axis=0).round(3),
    'def_lo':   np.percentile(defn, 5,  axis=0).round(3),
    'def_hi':   np.percentile(defn, 95, axis=0).round(3),
})

ratings['net'] = (ratings['att_mean'] - ratings['def_mean']).round(3)
ratings = ratings.sort_values('net', ascending=False).reset_index(drop=True)
ratings

,team,att_mean,att_sd,att_lo,att_hi,def_mean,def_sd,def_lo,def_hi,net
0,AGF,0.205,0.159,-0.073,0.463,-0.305,0.188,-0.620,-0.003,0.510
1,FC Midtjylland,0.277,0.151,0.024,0.523,-0.205,0.183,-0.513,0.087,0.482
2,Brøndby IF,0.013,0.168,-0.267,0.281,-0.269,0.179,-0.566,0.014,0.282
3,FC København,0.151,0.156,-0.111,0.406,-0.072,0.172,-0.362,0.203,0.223
4,Viborg,0.056,0.165,-0.217,0.315,-0.056,0.166,-0.329,0.213,0.112
5,Nordsjælland,0.092,0.160,-0.174,0.350,0.032,0.163,-0.237,0.292,0.060
6,Sønderjyske,-0.054,0.178,-0.352,0.228,0.041,0.162,-0.229,0.299,-0.095
7,OB,0.011,0.165,-0.268,0.278,0.134,0.160,-0.134,0.393,-0.123
8,Randers FC,-0.207,0.183,-0.519,0.083,-0.067,0.170,-0.356,0.201,-0.140
9,Fredericia,-0.104,0.179,-0.404,0.185,0.309,0.149,0.060,0.546,-0.413


In [24]:
# ── Expected goals vs average opponent ───────────────────────────────────
n_samp   = 10_000
idx      = np.random.choice(len(base), size=n_samp, replace=True)
team_idx = {name: i for i, name in enumerate(team_names)}

rows = []
for team in team_names:
    ti = team_idx[team]
    rows.append({
        'team':        team,
        'xg_home_for': round(np.exp(base[idx] + att[idx, ti] + hadv[idx]).mean(), 3),
        'xg_home_ag':  round(np.exp(base[idx] + defn[idx, ti]).mean(), 3),
        'xg_away_for': round(np.exp(base[idx] + att[idx, ti]).mean(), 3),
        'xg_away_ag':  round(np.exp(base[idx] + defn[idx, ti] + hadv[idx]).mean(), 3),
    })

xg_table = pd.DataFrame(rows)
xg_table['xg_diff'] = (
    xg_table['xg_home_for'] + xg_table['xg_away_for'] -
    xg_table['xg_home_ag']  - xg_table['xg_away_ag']
).round(3)
xg_table.sort_values('xg_diff', ascending=False).reset_index(drop=True)

,team,xg_home_for,xg_home_ag,xg_away_for,xg_away_ag,xg_diff
0,FC Midtjylland,1.929,1.044,1.685,1.195,1.375
1,AGF,1.797,0.948,1.569,1.085,1.333
2,Brøndby IF,1.485,0.981,1.298,1.123,0.679
3,FC København,1.699,1.188,1.484,1.360,0.635
4,Viborg,1.552,1.209,1.355,1.384,0.314
5,Nordsjælland,1.606,1.320,1.403,1.512,0.177
6,Sønderjyske,1.392,1.334,1.215,1.527,-0.254
7,Randers FC,1.194,1.200,1.044,1.373,-0.335
8,OB,1.481,1.463,1.294,1.676,-0.364
9,Vejle Boldklub,1.148,1.574,1.003,1.804,-1.227


In [25]:
# ── Fetch remaining Superligaen fixtures from TheSportsDB ────────────────────
import requests
import time
from scipy.stats import poisson
from datetime import date as _date_cls
from src.simulation import load_actual_results, run_multiple_seasons

_SPORTSDB_TO_MODEL = {
    'AGF Aarhus':       'AGF',
    'Brøndby':          'Brøndby IF',
    'FC Copenhagen':    'FC København',
    'FC Midtjylland':   'FC Midtjylland',
    'Fredericia':       'Fredericia',
    'FC Nordsjælland':  'Nordsjælland',
    'Odense BK':        'OB',
    'Randers FC':       'Randers FC',
    'Silkeborg IF':     'Silkeborg',
    'Sønderjyske':      'Sønderjyske',
    'Vejle':            'Vejle Boldklub',
    'Viborg':           'Viborg',
}

_SPORTSDB_ID = '4340'
_SEASON      = '2025-2026'

def _get_all_remaining_fixtures(start_round=1, end_round=40):
    fixtures = []
    for rnd in range(start_round, end_round + 1):
        try:
            r = requests.get(
                f'https://www.thesportsdb.com/api/v1/json/3/eventsround.php'
                f'?id={_SPORTSDB_ID}&r={rnd}&s={_SEASON}',
                timeout=10,
            )
            events = r.json().get('events') or []
            time.sleep(0.5)
        except Exception:
            continue
        for e in events:
            if e.get('intHomeScore') is None:
                home = _SPORTSDB_TO_MODEL.get(e['strHomeTeam'], e['strHomeTeam'])
                away = _SPORTSDB_TO_MODEL.get(e['strAwayTeam'], e['strAwayTeam'])
                fixtures.append({
                    'date': e['dateEvent'],
                    'time': (e.get('strTime') or '')[:5],
                    'gw':   e['intRound'],
                    'home': home,
                    'away': away,
                })
    return sorted(fixtures, key=lambda x: (x['date'], x['time']))

# ── Find the last completed round in the DB ───────────────────────────────────
import sqlite3 as _sq
_conn_r = _sq.connect(DB_PATH)
_max_round = _conn_r.execute(
    "SELECT MAX(CAST(round AS INTEGER)) FROM matches "
    "WHERE league_id=? AND season=? AND home_goals IS NOT NULL",
    [LEAGUE, SEASON]
).fetchone()[0] or 1
_conn_r.close()
_start_round = max(1, _max_round)
print(f"Fetching from official round {_start_round} onwards")

remaining_fixtures = _get_all_remaining_fixtures(start_round=_start_round)
print(f"Fetched {len(remaining_fixtures)} remaining fixtures")

unknown = set()
for f in remaining_fixtures:
    if f['home'] not in team_mapping: unknown.add(f['home'])
    if f['away'] not in team_mapping: unknown.add(f['away'])
if unknown:
    print(f"WARNING — unknown team names (add to _SPORTSDB_TO_MODEL): {unknown}")
remaining_fixtures[:10]

Fetching from official round 27 onwards
Fetched 30 remaining fixtures


[{'date': '2026-04-22',
  'time': '16:00',
  'gw': '28',
  'home': 'FC København',
  'away': 'OB'},
 {'date': '2026-04-22',
  'time': '16:00',
  'gw': '28',
  'home': 'Vejle Boldklub',
  'away': 'Silkeborg'},
 {'date': '2026-04-22',
  'time': '18:00',
  'gw': '28',
  'home': 'Viborg',
  'away': 'Brøndby IF'},
 {'date': '2026-04-23',
  'time': '16:00',
  'gw': '28',
  'home': 'Randers FC',
  'away': 'Fredericia'},
 {'date': '2026-04-23',
  'time': '16:00',
  'gw': '28',
  'home': 'Nordsjælland',
  'away': 'AGF'},
 {'date': '2026-04-23',
  'time': '18:00',
  'gw': '28',
  'home': 'Sønderjyske',
  'away': 'FC Midtjylland'},
 {'date': '2026-04-26',
  'time': '12:00',
  'gw': '29',
  'home': 'Fredericia',
  'away': 'OB'},
 {'date': '2026-04-26',
  'time': '12:00',
  'gw': '29',
  'home': 'Silkeborg',
  'away': 'Randers FC'},
 {'date': '2026-04-26',
  'time': '14:00',
  'gw': '29',
  'home': 'Sønderjyske',
  'away': 'Brøndby IF'},
 {'date': '2026-04-26',
  'time': '16:00',
  'gw': '29',
  'h

In [26]:
# ── Fixture predictions ───────────────────────────────────────────────────────
posterior = trace.posterior
_att  = posterior['att_str'].values.reshape(-1, n_teams)
_defn = posterior['def_str'].values.reshape(-1, n_teams)
_hadv = posterior['home_adv'].values.reshape(-1)
_base = posterior['baseline'].values.reshape(-1)
_rng  = np.random.choice(len(_base), size=2000, replace=True)

def _predict(home_team, away_team):
    hi = team_mapping[home_team]
    ai = team_mapping[away_team]
    home_lam = np.mean(np.exp(_base[_rng] + _hadv[_rng] + _att[_rng, hi] + _defn[_rng, ai])) + BASELINE_HOME_PENS
    away_lam = np.mean(np.exp(_base[_rng] + _att[_rng, ai] + _defn[_rng, hi])) + BASELINE_AWAY_PENS
    return home_lam, away_lam

def _match_probs(home_lam, away_lam, max_goals=10):
    h = np.array([poisson.pmf(g, home_lam) for g in range(max_goals + 1)])
    a = np.array([poisson.pmf(g, away_lam) for g in range(max_goals + 1)])
    grid = np.outer(h, a)
    return float(np.tril(grid, -1).sum()), float(np.trace(grid)), float(np.triu(grid, 1).sum())

rows = []
for f in remaining_fixtures:
    if f['home'] not in team_mapping or f['away'] not in team_mapping:
        continue
    hl, al = _predict(f['home'], f['away'])
    w, d, l = _match_probs(hl, al)
    rows.append({
        'GW':      f['gw'],
        'Date':    f['date'],
        'Time':    f['time'],
        'Home':    f['home'],
        'Away':    f['away'],
        'Home xG': round(hl, 2),
        'Away xG': round(al, 2),
        'Home W%': round(w * 100, 1),
        'Draw%':   round(d * 100, 1),
        'Away W%': round(l * 100, 1),
    })

_PRED_COLS = ['GW','Date','Time','Home','Away','Home xG','Away xG','Home W%','Draw%','Away W%']
pred_df = pd.DataFrame(rows, columns=_PRED_COLS) if rows else pd.DataFrame(columns=_PRED_COLS)
pred_df

,GW,Date,Time,Home,Away,Home xG,Away xG,Home W%,Draw%,Away W%
0,28,2026-04-22,16:00,FC København,OB,2.10,1.33,55.2,20.9,23.9
1,28,2026-04-22,16:00,Vejle Boldklub,Silkeborg,1.63,1.43,42.5,23.9,33.6
2,28,2026-04-22,18:00,Viborg,Brøndby IF,1.34,1.34,37.0,25.9,37.1
3,28,2026-04-23,16:00,Randers FC,Fredericia,1.78,1.20,51.1,23.3,25.6
4,28,2026-04-23,16:00,Nordsjælland,AGF,1.34,1.75,29.4,23.4,47.2
5,28,2026-04-23,18:00,Sønderjyske,FC Midtjylland,1.30,1.88,26.4,22.5,51.1
6,29,2026-04-26,12:00,Fredericia,OB,1.67,1.89,34.7,22.0,43.4
7,29,2026-04-26,12:00,Silkeborg,Randers FC,1.29,1.45,33.6,25.5,40.9
8,29,2026-04-26,14:00,Sønderjyske,Brøndby IF,1.23,1.47,31.6,25.6,42.8
9,29,2026-04-26,16:00,Viborg,Nordsjælland,1.77,1.44,45.4,23.1,31.5


In [27]:
# ── Season simulation using actual remaining fixtures ─────────────────────────
N_SIMS    = 5_000
df_actual = load_actual_results(DB_PATH, LEAGUE, SEASON)
print(f'Actual matches: {len(df_actual)}')

avg_table, position_freq = run_multiple_seasons(
    N_SIMS, trace, team_mapping, df_actual,
    remaining_fixtures=remaining_fixtures,
)

# ── Current actual points ─────────────────────────────────────────────────────
_teams = list(team_mapping.keys())
_pts   = {t: 0 for t in _teams}
_gd    = {t: 0 for t in _teams}

for _, r in df_actual[df_actual['home_goals'].notna()].iterrows():
    h, a = r['home_team'], r['away_team']
    if h not in _pts or a not in _pts:
        continue
    hg, ag = int(r['home_goals']), int(r['away_goals'])
    _gd[h] += hg - ag;  _gd[a] += ag - hg
    if   hg > ag: _pts[h] += 3
    elif ag > hg: _pts[a] += 3
    else:         _pts[h] += 1; _pts[a] += 1

avg_table = avg_table.merge(
    pd.DataFrame({'team': list(_pts.keys()), 'current_pts': list(_pts.values())}),
    on='team',
)
avg_table

Actual matches: 162
Pre-computing xG for 132 unique pairs...
  Played: 162   To simulate: 30
Running 5,000 simulations...
  2,000 / 5,000
  4,000 / 5,000


,team,avg_points,pts_low,pts_high,avg_wins,avg_draws,avg_losses,avg_goals_for,avg_goals_against,avg_xg_for,avg_xg_against,avg_position,title_pct,top5_pct,top8_pct,relegation_pct,avg_goal_difference,avg_xg_difference,current_pts
0,AGF,64.4974,60.775352,68.219448,18.4316,9.2026,4.3658,59.4734,34.0310,57.990812,36.893365,1.3522,64.8,100.0,100.0,0.0,25.4424,21.097447,56
1,FC Midtjylland,62.1626,58.395362,65.929838,17.3314,10.1684,4.5002,73.6346,35.6864,61.465544,39.990725,1.6500,35.2,100.0,100.0,0.0,37.9482,21.474819,54
2,Nordsjælland,49.8176,46.120298,53.514902,15.5434,3.1874,13.2692,51.8768,50.5568,50.426344,50.889690,3.9426,0.0,88.3,100.0,0.0,1.3200,-0.463346,44
3,FC København,47.5764,43.922552,51.230248,13.8386,6.0606,12.1008,60.2742,46.0320,58.803965,42.925738,4.6108,0.0,75.6,99.5,0.0,14.2422,15.878227,38
4,Viborg,46.5454,42.900492,50.190308,13.7938,5.1640,13.0422,50.1356,47.7576,49.262404,47.122890,5.2746,0.0,56.8,99.4,0.0,2.3780,2.139514,40
5,Brøndby IF,44.9188,41.209139,48.628461,12.8782,6.2842,12.8376,45.8156,33.6714,48.396842,38.962808,5.9080,0.0,39.8,98.2,0.0,12.1442,9.434033,38
6,OB,44.3806,40.640602,48.120598,12.0948,8.0962,11.8090,52.4804,59.8006,50.470368,52.858868,6.5712,0.0,23.4,92.8,0.0,-7.3202,-2.388500,37
7,Sønderjyske,43.4620,39.895654,47.028346,11.4254,9.1858,11.3888,43.2784,49.5112,44.061054,52.171309,6.9336,0.0,16.1,93.1,0.0,-6.2328,-8.110255,38
8,Randers FC,37.5384,33.841055,41.235745,10.1056,7.2216,14.6728,35.4070,42.5070,42.285979,44.918802,9.1572,0.0,0.1,14.3,0.0,-7.1000,-2.632823,30
9,Fredericia,34.9958,31.298797,38.692803,9.6264,6.1166,16.2570,43.5458,67.0136,44.354520,63.039451,10.0246,0.0,0.0,2.5,0.0,-23.4678,-18.684931,29


In [28]:
# ── Split into championship and relegation groups ─────────────────────────────
from collections import defaultdict

# Infer groups from remaining fixtures via connected components.
# Teams that play each other in the playoffs are in the same group.
adj = defaultdict(set)
playoff_teams = set()
for f in remaining_fixtures:
    if f['home'] in team_mapping and f['away'] in team_mapping:
        adj[f['home']].add(f['away'])
        adj[f['away']].add(f['home'])
        playoff_teams.update([f['home'], f['away']])

visited, groups = set(), []
for team in sorted(playoff_teams):
    if team not in visited:
        component, queue = set(), [team]
        while queue:
            t = queue.pop()
            if t not in visited:
                visited.add(t); component.add(t)
                queue.extend(adj[t] - visited)
        groups.append(component)

# Championship group has higher current points on average
pts_map = dict(zip(avg_table['team'], avg_table['current_pts']))
groups.sort(key=lambda g: -sum(pts_map.get(t, 0) for t in g))
champ_group = groups[0] if groups     else set()
rel_group   = groups[1] if len(groups) > 1 else set()

print(f"Championship group: {sorted(champ_group)}")
print(f"Relegation group:   {sorted(rel_group)}")

_COLS = ['team', 'current_pts', 'avg_points', 'avg_wins', 'avg_draws',
         'avg_losses', 'avg_goal_difference']

print("\n── Championship Group ─────────────────────────────────────────")
champ_df = (avg_table[avg_table['team'].isin(champ_group)]
            .sort_values('current_pts', ascending=False)
            [_COLS].reset_index(drop=True))
display(champ_df)

print("\n── Relegation Group ───────────────────────────────────────────")
rel_df = (avg_table[avg_table['team'].isin(rel_group)]
          .sort_values('current_pts', ascending=False)
          [_COLS].reset_index(drop=True))
display(rel_df)

Championship group: ['AGF', 'Brøndby IF', 'FC Midtjylland', 'Nordsjælland', 'Sønderjyske', 'Viborg']
Relegation group:   ['FC København', 'Fredericia', 'OB', 'Randers FC', 'Silkeborg', 'Vejle Boldklub']

── Championship Group ─────────────────────────────────────────


,team,current_pts,avg_points,avg_wins,avg_draws,avg_losses,avg_goal_difference
0,AGF,56,64.4974,18.4316,9.2026,4.3658,25.4424
1,FC Midtjylland,54,62.1626,17.3314,10.1684,4.5002,37.9482
2,Nordsjælland,44,49.8176,15.5434,3.1874,13.2692,1.3200
3,Viborg,40,46.5454,13.7938,5.1640,13.0422,2.3780
4,Brøndby IF,38,44.9188,12.8782,6.2842,12.8376,12.1442
5,Sønderjyske,38,43.4620,11.4254,9.1858,11.3888,-6.2328



── Relegation Group ───────────────────────────────────────────


,team,current_pts,avg_points,avg_wins,avg_draws,avg_losses,avg_goal_difference
0,FC København,38,47.5764,13.8386,6.0606,12.1008,14.2422
1,OB,37,44.3806,12.0948,8.0962,11.8090,-7.3202
2,Randers FC,30,37.5384,10.1056,7.2216,14.6728,-7.1000
3,Fredericia,29,34.9958,9.6264,6.1166,16.2570,-23.4678
4,Silkeborg,27,32.5248,8.4586,7.1490,16.3924,-25.2058
5,Vejle Boldklub,18,23.5928,4.4848,10.1384,17.3768,-24.1484


In [29]:
# ── AGF vs FC Midtjylland — xP from remaining fixtures ───────────────────────
_FOCUS = ['AGF', 'FC Midtjylland']

rows = []
for f in remaining_fixtures:
    h, a = f['home'], f['away']
    if h not in _FOCUS and a not in _FOCUS:
        continue
    if h not in team_mapping or a not in team_mapping:
        continue
    hl, al  = _predict(h, a)
    w, d, l = _match_probs(hl, al)

    for team in _FOCUS:
        if team not in (h, a):
            continue
        is_home = (team == h)
        win_p   = w if is_home else l
        loss_p  = l if is_home else w
        xp      = round(3 * win_p + d, 3)
        rows.append({
            'Team':     team,
            'GW':       f['gw'],
            'Date':     f['date'],
            'Venue':    'H' if is_home else 'A',
            'Opponent': a if is_home else h,
            'xG For':   round(hl if is_home else al, 2),
            'xG Ag':    round(al if is_home else hl, 2),
            'W%':       round(win_p  * 100, 1),
            'D%':       round(d      * 100, 1),
            'L%':       round(loss_p * 100, 1),
            'xP':       xp,
        })

xp_df = pd.DataFrame(rows).sort_values(['Team', 'Date']).reset_index(drop=True)

for team in _FOCUS:
    t_df     = xp_df[xp_df['Team'] == team].reset_index(drop=True)
    cur_pts  = pts_map.get(team, 0)
    total_xp = round(t_df['xP'].sum(), 1)
    print(f"{'─'*58}")
    print(f"  {team}  |  current: {cur_pts} pts  |  xP remaining: {total_xp}  |  projected: {cur_pts + total_xp:.1f}")
    print(f"{'─'*58}")
    display(t_df.drop(columns='Team'))
    print()

──────────────────────────────────────────────────────────
  AGF  |  current: 56 pts  |  xP remaining: 8.5  |  projected: 64.5
──────────────────────────────────────────────────────────


,GW,Date,Venue,Opponent,xG For,xG Ag,W%,D%,L%,xP
0,28,2026-04-23,A,Nordsjælland,1.75,1.34,47.2,23.4,29.4,1.651
1,29,2026-04-26,H,FC Midtjylland,1.63,1.37,43.6,24.1,32.3,1.548
2,30,2026-05-03,H,Sønderjyske,2.03,1.02,60.9,20.9,18.2,2.037
3,31,2026-05-10,A,Brøndby IF,1.32,1.25,38.4,26.5,35.1,1.416
4,32,2026-05-17,H,Viborg,1.86,1.12,54.7,22.6,22.6,1.868



──────────────────────────────────────────────────────────
  FC Midtjylland  |  current: 54 pts  |  xP remaining: 8.2  |  projected: 62.2
──────────────────────────────────────────────────────────


,GW,Date,Venue,Opponent,xG For,xG Ag,W%,D%,L%,xP
0,28,2026-04-23,A,Sønderjyske,1.88,1.30,51.1,22.5,26.4,1.759
1,29,2026-04-26,A,AGF,1.37,1.63,32.3,24.1,43.6,1.211
2,30,2026-05-04,H,Viborg,1.99,1.22,55.2,21.6,23.1,1.874
3,31,2026-05-10,A,Nordsjælland,1.88,1.46,47.5,22.4,30.2,1.647
4,32,2026-05-17,H,Brøndby IF,1.64,1.18,48.2,24.4,27.4,1.689


In [30]:
# ── Datawrapper export — GW-aligned xP table ─────────────────────────────────
_T1, _T2 = 'AGF', 'FC Midtjylland'
_t1 = xp_df[xp_df['Team'] == _T1].set_index('GW')
_t2 = xp_df[xp_df['Team'] == _T2].set_index('GW')

gws = sorted(set(_t1.index) | set(_t2.index))
dw_rows = []
for gw in gws:
    r1 = _t1.loc[gw] if gw in _t1.index else None
    r2 = _t2.loc[gw] if gw in _t2.index else None
    date = r1['Date'] if r1 is not None else r2['Date']
    dw_rows.append({
        'GW':          gw,
        'Date':        date,
        f'{_T1} game': f"{'H' if r1['Venue']=='H' else 'A'} {r1['Opponent']}" if r1 is not None else '',
        f'{_T1} xP':   r1['xP'] if r1 is not None else '',
        f'{_T2} game': f"{'H' if r2['Venue']=='H' else 'A'} {r2['Opponent']}" if r2 is not None else '',
        f'{_T2} xP':   r2['xP'] if r2 is not None else '',
    })

dw_df = pd.DataFrame(dw_rows)

_t1_cur = pts_map.get(_T1, 0)
_t2_cur = pts_map.get(_T2, 0)
_t1_xp  = round(xp_df[xp_df['Team']==_T1]['xP'].sum(), 1)
_t2_xp  = round(xp_df[xp_df['Team']==_T2]['xP'].sum(), 1)
dw_df.loc[len(dw_df)] = {
    'GW': 'Projected',
    'Date': '',
    f'{_T1} game': f'Current: {_t1_cur} pts',
    f'{_T1} xP':   round(_t1_cur + _t1_xp, 1),
    f'{_T2} game': f'Current: {_t2_cur} pts',
    f'{_T2} xP':   round(_t2_cur + _t2_xp, 1),
}

print(dw_df.to_csv(index=False))

GW,Date,AGF game,AGF xP,FC Midtjylland game,FC Midtjylland xP
28,2026-04-23,A Nordsjælland,1.651,A Sønderjyske,1.759
29,2026-04-26,H FC Midtjylland,1.548,A AGF,1.211
30,2026-05-03,H Sønderjyske,2.037,H Viborg,1.874
31,2026-05-10,A Brøndby IF,1.416,A Nordsjælland,1.647
32,2026-05-17,H Viborg,1.868,H Brøndby IF,1.689
Projected,,Current: 56 pts,64.5,Current: 54 pts,62.2

